In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

const bb = 120
const aa = 40
const N = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter_per_k = 20000
const KMAX_UPPER    = 30

include("functions.jl")

Random.seed!(2025)

model_tag_sym = :sliding
Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = String[]
if model_tag_sym === :memoryless
    header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    header_cont = ["beta", "alpha", "gamma", "lambda_R"]
elseif model_tag_sym === :sliding
    header_cont = ["beta", "alpha", "gamma"]
end

c = 1

t_all = Dates.now()
for k in 1:KMAX_UPPER
    Random.seed!(2025 + c*10_000 + k)
    initθ_chain = initθ_for_chain(model_tag_sym)

    t0 = Dates.now()
    try
        samples, loglik_aug_vecs =
            mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
                fit_mech      = model_tag_sym,
                n_iter        = n_iter_per_k,
                initθ         = initθ_chain,
                KMAX_UPPER    = KMAX_UPPER,
                k_max_fixed   = k)

        if size(samples, 1) == 0
            error("Empty samples returned for k=$k.")
        end

        samples_filename = "samples_chain_$(c)_k$(k).csv"
        write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

        loglik_filename = "loglik_chain_$(c)_k$(k).csv"
        write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

        el = Dates.value(Dates.now() - t0) / 1000
        @info(@sprintf("Chain %d, k=%d: ok in %.2fs", c, k, el))
    catch err
        el = Dates.value(Dates.now() - t0) / 1000
        @warn(@sprintf("Chain %d, k=%d: ERROR - %s after %.2fs", c, k, err, el))
        continue
    end
end

el_all = Dates.value(Dates.now() - t_all) / 1000
@info(@sprintf("All k (1..%d) finished. Total time %.2fs -> output dir: %s",
               KMAX_UPPER, el_all, out_dir))


[ Info: [sliding] iter 1000/20000 elapsed=5.3s, rate=0.159, mean=[0.613, 0.01025, 0.809], std=[0.0651, 0.003135, 0.1113] [ADAPT]
[ Info: [sliding] iter 2000/20000 elapsed=11.1s, rate=0.155, mean=[0.623, 0.00660, 0.852], std=[0.0473, 0.003985, 0.0878] [ADAPT]
[ Info: [sliding] iter 3000/20000 elapsed=15.0s, rate=0.147, mean=[0.641, 0.00473, 0.857], std=[0.0459, 0.004028, 0.0730] [ADAPT]
[ Info: [sliding] iter 4000/20000 elapsed=18.9s, rate=0.140, mean=[0.654, 0.00371, 0.851], std=[0.0447, 0.003829, 0.0645] [ADAPT]
[ Info: [sliding] iter 5000/20000 elapsed=22.8s, rate=0.135, mean=[0.665, 0.00309, 0.848], std=[0.0445, 0.003600, 0.0581] [ADAPT]
[ Info: [sliding] iter 6000/20000 elapsed=26.7s, rate=0.136, mean=[0.665, 0.00274, 0.848], std=[0.0411, 0.003363, 0.0533] [ADAPT]
[ Info: [sliding] iter 7000/20000 elapsed=30.5s, rate=0.135, mean=[0.671, 0.00246, 0.855], std=[0.0413, 0.003173, 0.0519] [ADAPT]
[ Info: [sliding] iter 8000/20000 elapsed=34.4s, rate=0.133, mean=[0.683, 0.00222, 0.864], 

In [ ]:
using DelimitedFiles
using Statistics
using Random
using Printf

const OUTPUT_DIR        = "output"
const BURN_IN_LOGLIK    = 0
const THIN_LOGLIK       = 1

function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function detect_k_values(outdir::String)
    files = filter(isfile, readdir(outdir; join=true))
    ks = Int[]
    for f in files
        m = match(r"loglik_chain_(\d+)_k(\d+)\.csv", f)
        if m !== nothing
            push!(ks, parse(Int, m.captures[2]))
        end
    end
    return unique(sort(ks))
end

function load_loglik_for_k(k::Int, outdir::String)
    path = joinpath(outdir, "loglik_chain_1_k$(k).csv")
    if !isfile(path)
        @warn "Missing loglik file: $path"
        return Array{Float64}(undef, 0, 0)
    end
    M, _ = read_csv_matrix(path)
    if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
        idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
        M = M[idxs, :]
    end
    return M
end

# ========= WAIC =========
function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    for j in 1:T
        col = L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic_from_matrix(L::AbstractMatrix{<:Real})
    if isempty(L)
        return NaN
    end
    if any(isinf, L)
        @warn "Encountered infinite log-likelihoods. WAIC will be -Inf."
        return -Inf
    end
    lppd = sum(logmeanexp_columnwise(L))
    pwaic = sum(var(L, dims=1))
    return -2 * lppd + 2 * pwaic
end

function write_waic_k_csv(path::String, rows::Vector{Tuple{Int,Float64}})
    open(path, "w") do io
        println(io, "kmax,WAIC")
        for (k, w) in rows
            println(io, "$k,$w")
        end
    end
end

function write_best3_csv(path::String, vec::Vector{Tuple{Int,Float64}})
    open(path, "w") do io
        println(io, "kmin1,WAIC1,kmin2,WAIC2,kmin3,WAIC3")
        k1, w1 = vec[1]
        k2, w2 = length(vec) >= 2 ? vec[2] : (NaN, NaN)
        k3, w3 = length(vec) >= 3 ? vec[3] : (NaN, NaN)
        println(io, "$k1,$w1,$k2,$w2,$k3,$w3")
    end
end

Random.seed!(2025)

ks = detect_k_values(OUTPUT_DIR)
isempty(ks) && error("No loglik_chain_1_k*.csv found in $(OUTPUT_DIR)")

waics = Tuple{Int,Float64}[]  # (k, waic)

@info "Processing single dataset ... (k = $(first(ks))..$(last(ks)))"
for k in ks
    L = load_loglik_for_k(k, OUTPUT_DIR)
    if size(L,1) == 0
        @warn "Empty loglik for k=$k"
        push!(waics, (k, NaN))
        continue
    end
    w = compute_waic_from_matrix(L)
    push!(waics, (k, w))
end

valid = filter(x -> !isnan(x[2]), waics)
isempty(valid) && error("All WAIC are NaN.")

sorted = sort(valid; by = x -> x[2])
best3  = sorted[1:min(3, length(sorted))]

write_waic_k_csv(joinpath(OUTPUT_DIR, "waic_by_k.csv"), waics)
write_best3_csv(joinpath(OUTPUT_DIR, "best3.csv"), best3)

@info "Done. Outputs written in $(OUTPUT_DIR): waic_by_k.csv, best3.csv"


[ Info: Processing single dataset ... (k = 1..30)
[ Info: Done. Outputs written in output: waic_by_k.csv, best3.csv
